# Data Science and Machine Learning

On Friday we'll use pandas and scikit-learn on real data. Before the libraries, the ideas, in
plain Python with nothing but lists, dicts and `csv`. The data: the 891 passengers of the Titanic
whose fate is known.

In [ ]:
import csv
import random
from pathlib import Path

DATA = Path("./ppp-2026/data/titanic.csv")

with open(DATA, encoding="utf-8") as file:
    passengers = list(csv.DictReader(file))

print(len(passengers))
passengers

Every value is a string, including `Survived` (`"0"` or `"1"`). The names contain commas
(`"Braund, Mr. Owen Harris"`): that's why we use `csv` and not `split(",")` (week 4).


## The words you'll hear

| Word | Here |
|---|---|
| **features** | what the model may look at: `Sex`, `Pclass`, `Age`, ... |
| **label**, target | what it should predict: `Survived` |
| **training**, fitting | `learn`: building the rule from examples |
| **model** | the result of training, used to **predict**: `as_model(rule, key)` |
| **classification** | predicting a category (survived or not); **regression**: a number (a fare, a price) |
| **supervised** | learning from examples with the right answer; **unsupervised**: finding groups without answers |
| **baseline** | the simplest model, the number to beat. |
| **accuracy** | fraction of right predictions; one **metric** of many |
| **train / test split** | learn on one part, measure on the part the model never saw |
| **overfitting** | memorizing the training data instead of learning something that carries over |
| **hyperparameter** | a choice made before training: here, which grouping |


# What does it mean that a program "learns"?

## A model is a function

A **model** takes what we know about a passenger and returns a guess: `"1"` survived, `"0"` didn't.
Here's the simplest one there is. **Predict:** how often is it right?

In [ ]:
def nobody_survives(passenger):
    return "0"


def accuracy(model, passengers):
    right = 0
    for passenger in passengers:
        if model(passenger) == passenger["Survived"]:
            right += 1
    return right / len(passengers)


print(accuracy(nobody_survives, passengers))

62% without looking at the passenger at all. That's the **baseline**: any model has to beat it to
be worth anything. "My model is 62% accurate" sounds fine until you know this number.

**Predict:** "Women and children first". A model that only looks at `Sex`?

In [ ]:
def women_survive(passenger):
    if passenger["Sex"] == "female":
        return "1"
    return "0"


print(accuracy(women_survive, passengers))

## Learning: let the data choose the rule

We wrote `women_survive` ourselves. **Learning** means: a program looks at examples and builds the
rule itself. Here's a very simple learner. It puts passengers into groups (by a `key` function),
and for each group remembers what happened to the majority. It's the dict accumulator from week 3.

In [ ]:
def learn(passengers, key):
    counts = {}
    for passenger in passengers:
        group = key(passenger)
        survived, total = counts.get(group, (0, 0))
        counts[group] = (survived + int(passenger["Survived"]), total + 1)
    rule = {}
    for group, (survived, total) in counts.items():
        if survived * 2 > total:
            rule[group] = "1"
        else:
            rule[group] = "0"
    return rule


def by_sex(passenger):
    return passenger["Sex"]


print(learn(passengers, by_sex))

It found `women_survive` by itself. To use the learned rule as a model, look the passenger's group up.
A group it has never seen gets `"0"`, the baseline.

In [ ]:
def as_model(rule, key):
    def model(passenger):
        return rule.get(key(passenger), "0")

    return model


model = as_model(learn(passengers, by_sex), by_sex)
print(accuracy(model, passengers))

Now finer groups: sex, and whether the passenger is a child (under 16). Some ages are missing
(`""`): they get their own group.

In [ ]:
def age_group(passenger):
    if passenger["Age"] == "":
        return "unknown"
    if float(passenger["Age"]) < 16:
        return "child"
    return "adult"


def by_sex_and_age(passenger):
    return passenger["Sex"], age_group(passenger)


rule = learn(passengers, by_sex_and_age)
print(rule)
model = as_model(rule, by_sex_and_age)
print(accuracy(model, passengers))

Boys under 16 now get `"1"`: "women and children first", found in the data. 21 of 40 boys
survived, so it's a close call, and the accuracy only goes up a little (0.787 to 0.789).

**Predict:** if finer groups are better, what about the finest grouping of all, one passenger per
group? How accurate is it?

In [ ]:
def by_name(passenger):
    return passenger["Name"]


model = as_model(learn(passengers, by_name), by_name)
print(accuracy(model, passengers))

100%! A perfect model? It has **memorized** every answer. It knows nothing about passengers it
hasn't seen.

## Train and test

The fair question: how good is a model on passengers it **hasn't** learned from? So we hide some.
Shuffle (with a fixed seed, so it's the same every time), learn from 700, test on the other 191.

In [ ]:
shuffled = passengers.copy()
random.Random(1).shuffle(shuffled)
train = shuffled[:700]
test = shuffled[700:]

print(len(train), len(test))
print("baseline on test:", round(accuracy(nobody_survives, test), 3))

**Predict:** the `by_name` model on the test passengers?

In [ ]:
model = as_model(learn(train, by_name), by_name)
print("train:", accuracy(model, train))
print("test: ", round(accuracy(model, test), 3))

Every test passenger is unknown, so it says `"0"` for all of them: exactly the baseline. Perfect
on what it learned from, useless on anything new. That's **overfitting**.

Now a whole range of groupings, from coarse to fine. Each line: how many groups the rule has, and
the accuracy on train and on test. **Predict:** which line is best on test?

In [ ]:
keys = {
    "sex": by_sex,
    "sex, child": by_sex_and_age,
    "sex, class, child": lambda p: (p["Sex"], p["Pclass"], age_group(p)),
    "sex, class, exact age": lambda p: (p["Sex"], p["Pclass"], p["Age"]),
    "sex, class, age, fare": lambda p: (p["Sex"], p["Pclass"], p["Age"], p["Fare"]),
    "name": by_name,
}

for label, key in keys.items():
    rule = learn(train, key)
    model = as_model(rule, key)
    print(
        f"{label:22} {len(rule):4} groups   "
        f"train {accuracy(model, train):.3f}   test {accuracy(model, test):.3f}"
    )

Train accuracy only ever goes up: more groups, more memorized. Test accuracy goes up a little, then
falls. Best on new data: "sex, child" and "sex, class, child", tied. When two models are
equally good on new data, take the simpler one, and "sex, child" is exactly "women and children
first".

**Predict:** with a different shuffle (`Random(2)`), are the numbers the same?

In [ ]:
shuffled = passengers.copy()
random.Random(2).shuffle(shuffled)
train = shuffled[:700]
test = shuffled[700:]

for label, key in keys.items():
    model = as_model(learn(train, key), key)
    print(
        f"{label:22} train {accuracy(model, train):.3f}   test {accuracy(model, test):.3f}"
    )

Different numbers, same shape: train goes up, test goes up a little and then down. This time
"sex" and "sex, child" tie for best on test, and class makes it worse. With 191 test passengers, one more right answer is half a percent:
differences of 1–2% between models may just be luck in the split.

Choosing the grouping by looking at the test numbers, as we just did, is cheating a little: the
test set helped make the choice. Properly, you'd split off a third part (**validation**) for
choosing, and look at the test set once, at the end.

On Friday, scikit-learn's `DecisionTreeClassifier` does what `learn` does, but finds the groups
itself: it picks the question that splits the passengers best (sex first), then the best next
question in each group, and so on. Its `max_depth` is our "how fine are the groups".

## When not to use machine learning

- **When a rule does the job.** "Women and children first" is about 80% and fits on a napkin. A model
  that's 81% and nobody can explain is rarely worth it.
- **When there's little data.** 891 passengers is small: one test passenger is half a percent.
- **When the data carries a past decision.** The model "learns" that third-class passengers drown.
  That's not a law of nature; it's how the ship was built and run in 1912. Trained on hiring or
  lending data, a model learns past decisions the same way, and repeats them.
- **When mistakes need an explanation.** Someone refused a loan can ask why. "The model said so"
  isn't an answer.
- **When the answer can be computed.** Nobody trains a model to add numbers, or to find the
  lowest-risk path through a cave.